# S09 pratimas

```
S9 pratimas -- kokybės vertinimas.

Užduotis
--------
1. Įgyvendinkite ``build_teaching_checklist()`` -- apie 6 elementus,
   atsakomus „yes" / „no" / „unclear".
2. Įgyvendinkite ``appraise(paper_text, checklist, *, call_llm)``:
   - paklauskite DKM kiekvieno elemento su įrodymo citata;
   - PAŽODŽIUI patikrinkite, ar citata yra tekste;
   - patvirtintas „no" -> ``provisional_pending_human_confirmation=True``
     (bet studija NEATMETAMA);
   - nepatvirtinta citata arba „unclear" be įrodymo -> suvienodinti į
     „unclear", ``provisional=False``, įrašyti į ``review_queue`` su
     priežastimi.
3. Įvertinkite ``papers/relevant.md``. Paaiškinkite skirtumą tarp
   provizorinio elemento (patvirtintas „no") ir sprendimų eilės elemento
   (nepatvirtinama), ir kodėl nė vienas neatmeta studijos automatiškai.

Paleidimas: ``python -m examples.chapter.s09_quality_appraisal.exercise``
```

1. Paleiskite paruošimo langelį. 2. Užpildykite `TODO` funkcijas pratimo langelyje ir paleiskite jį. 3. Palyginkite su pavyzdiniu sprendimu.

### Prieš pradedant

1. **Pirmas paleidimas.** Paleidus pirmą kodo langelį, Colab gali parodyti įspėjimą *„Notebook was not authored by Google“*. Tai normalu (sąsiuvinis atkeltas iš GitHub): spauskite **Run anyway**.
2. **Pirmas langelis** atsisiunčia kodą ir įdiegia bibliotekas (maždaug 1-2 min.). Paleiskite jį pirmą ir palaukite, kol pasirodys `Paruosta:`.
3. **Tikras DKM nebūtinas.** Jei norite jį išbandyti: kairėje Colab juostoje spauskite rakto piktogramą (*Secrets*), *Add new secret*, pavadinimas lygiai **`OPENAI_API_KEY`**, reikšmė -- jūsų OpenAI raktas, ir įjunkite *Notebook access*. Raktą rašykite tik čia, niekada į kodo langelį. Kreipiniai apmokami.

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/rimraul/slr-automation.git"
REPO_DIR = REPO_URL.rsplit("/", 1)[-1].removesuffix(".git")
if "google.colab" in sys.modules and not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
sys.path.insert(0, os.getcwd())
for req in ['examples/chapter/requirements-colab.txt']:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req],
                       capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise SystemExit("pip install nepavyko (zr. klaida aukščiau): " + req)
print("Paruosta:", os.getcwd())

In [ ]:

from __future__ import annotations

from typing import Literal

from pydantic import BaseModel

from examples.chapter import toolkit


class ChecklistItem(BaseModel):
    item_id: str
    question: str


class AppraisalItem(BaseModel):
    item_id: str
    status: Literal["yes", "no", "unclear"]
    provisional_pending_human_confirmation: bool
    evidence_quote: str


class QueueEntry(BaseModel):
    item_id: str
    queue_reason: str


class AppraisalResult(BaseModel):
    items: list[AppraisalItem]
    n_provisional: int
    review_queue: list[QueueEntry]
    requires_human_action: bool


def build_teaching_checklist() -> list[ChecklistItem]:
    """TODO: apie 6 kontrolinio sąrašo elementai."""
    raise NotImplementedError


def appraise(paper_text: str, checklist, *, call_llm=toolkit.call_llm) -> AppraisalResult:
    """TODO: vertinimas su pažodiniu įrodymų tikrinimu ir dviem keliais."""
    raise NotImplementedError


def main() -> None:
    paper = toolkit.load_fixture("papers/relevant.md")
    scripted = toolkit.scripted_llm(
        {"relevant": toolkit.SCRIPTED_RESPONSES["s09"]["papers"]["relevant"]},
        key_fn=lambda m: "relevant",
    )
    print("S9 quality appraisal exercise")
    try:
        result = appraise(paper, build_teaching_checklist(), call_llm=scripted)
    except NotImplementedError:
        print("appraise()/build_teaching_checklist() not implemented yet -- fill them in and re-run.")
        return
    print(f"provisional: {result.n_provisional}; queue: {len(result.review_queue)}")


main()

## Pavyzdinis sprendimas

Pažiūrėkite tik pabandę patys. Žemiau esantys testai tikrina **pavyzdinį sprendimą**, ne jūsų kodą.

In [ ]:
!cat examples/chapter/s09_quality_appraisal/solution.py

In [ ]:
!python -m pytest examples/chapter/s09_quality_appraisal -q

In [ ]:
# NEPRIVALOMA: tikri kreipiniai i DKM (sumoka uz naudojimą). Raktą laikykite
# Colab "Secrets" (rakto piktograma kairėje) pavadinimu OPENAI_API_KEY; niekada ne užrašykite kode.
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")